# Spare Part Catalog Search and Duplicate Checks

**Portfolio demo · synthetic data only**

This notebook demonstrates how bilingual search and duplicate checks can help staff work with a partial spare-parts catalog. All records, vendors and prices below are fictional.

## 1. Build a small synthetic catalog

The demo intentionally includes Indonesian/English descriptions and a few blank fields to represent information awaiting confirmation.

In [ ]:
import pandas as pd
from difflib import SequenceMatcher
import re

records = [
    {"pn":"DEMO-BAT-001","description_en":"Forklift battery 48V 600Ah","description_id":"Baterai forklift 48V 600Ah","category":"Battery","unit":"Forklift A","vendor":"Demo Vendor North","brand":"Voltix","net_price":12500000,"buyer_tier":"Standard"},
    {"pn":"DEMO-BAT-002","description_en":"Forklift battery 48V 750Ah","description_id":"Baterai forklift 48V 750Ah","category":"Battery","unit":"Forklift B","vendor":"Demo Vendor East","brand":"Voltix","net_price":14950000,"buyer_tier":"Standard"},
    {"pn":"DEMO-TIR-001","description_en":"Solid tire 18x7-8","description_id":"Ban solid 18x7-8","category":"Tire","unit":"Forklift A","vendor":"Demo Vendor West","brand":"Roadmark","net_price":1850000,"buyer_tier":"Tier A"},
    {"pn":"DEMO-TIR-002","description_en":"Pneumatic tire 6.00-9","description_id":"Ban pneumatic 6.00-9","category":"Tire","unit":"Forklift C","vendor":"Demo Vendor East","brand":"Roadmark","net_price":2150000,"buyer_tier":"Tier B"},
    {"pn":"DEMO-FIL-001","description_en":"Hydraulic oil filter","description_id":"Filter oli hidrolik","category":"Filter","unit":"Forklift A","vendor":"Demo Vendor North","brand":"ClearFlow","net_price":425000,"buyer_tier":"Standard"},
    {"pn":"DEMO-FIL-002","description_en":"Air filter element","description_id":"Elemen filter udara","category":"Filter","unit":"Forklift B","vendor":"Demo Vendor South","brand":"ClearFlow","net_price":310000,"buyer_tier":"Standard"},
    {"pn":"DEMO-BRK-001","description_en":"Brake shoe assembly","description_id":"Sepatu rem assembly","category":"Brake","unit":"Forklift C","vendor":"Demo Vendor South","brand":"Stopwell","net_price":980000,"buyer_tier":"Tier A"},
    {"pn":"DEMO-BRK-002","description_en":"Brake cylinder kit","description_id":"Kit silinder rem","category":"Brake","unit":"Forklift A","vendor":"Demo Vendor West","brand":"Stopwell","net_price":760000,"buyer_tier":"Tier B"},
    {"pn":"DEMO-BRG-001","description_en":"Mast roller bearing","description_id":"Bantalan roller mast","category":"Bearing","unit":"Forklift B","vendor":"Demo Vendor North","brand":"RollPro","net_price":670000,"buyer_tier":"Standard"},
    {"pn":"DEMO-BRG-002","description_en":"Wheel bearing set","description_id":"Set bantalan roda","category":"Bearing","unit":"Forklift C","vendor":"Demo Vendor East","brand":"RollPro","net_price":590000,"buyer_tier":"Standard"},
    {"pn":"DEMO-HOS-001","description_en":"Hydraulic hose 1/2 inch","description_id":"Selang hidrolik 1/2 inci","category":"Hydraulic","unit":"Forklift A","vendor":"Demo Vendor West","brand":"Flexline","net_price":450000,"buyer_tier":"Tier A"},
    {"pn":"DEMO-SEN-001","description_en":"Seat switch assembly","description_id":"Saklar kursi assembly","category":"Electrical","unit":"Forklift B","vendor":"","brand":"Brightway","net_price":None,"buyer_tier":"Standard"},
    {"pn":"DEMO-ALT-001","description_en":"Alternator 12V 45A","description_id":"Alternator 12V 45A","category":"Electrical","unit":"Forklift C","vendor":"Demo Vendor East","brand":"Brightway","net_price":2350000,"buyer_tier":"Tier A"},
]
df = pd.DataFrame(records)
df.head()

## 2. Normalize wording and add bilingual aliases

The first version uses a transparent token-and-alias method so results are explainable. Fuzzy similarity helps surface likely typos; staff still verify the returned part before quoting a price.

In [ ]:
ALIASES = {
    "battery": {"battery", "batteries", "baterai", "aki"},
    "tire": {"tire", "tires", "tyre", "tyres", "ban"},
    "filter": {"filter", "saringan"},
    "brake": {"brake", "rem"},
}

def normalize(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value or "").lower()).strip()

def expand_terms(query):
    tokens = normalize(query).split()
    expanded = set(tokens)
    for token in tokens:
        for terms in ALIASES.values():
            if token in terms or any(len(token) >= 3 and (token in term or term in token) for term in terms):
                expanded.update(terms)
    return expanded

def search_catalog(data, query, threshold=0.55):
    terms = expand_terms(query)
    fields = ["pn", "description_en", "description_id", "category", "unit", "vendor", "brand"]
    scores=[]
    for _, row in data.iterrows():
        text = normalize(" ".join(str(row.get(field, "")) for field in fields))
        words=set(text.split())
        score=0.0
        for term in terms:
            if term in words:
                score=max(score, 1.0)
            elif words:
                score=max(score, max(SequenceMatcher(None, term, word).ratio() for word in words))
        scores.append(score)
    out=data.copy();out["match_score"]=scores
    return out[out.match_score>=threshold].sort_values("match_score", ascending=False)

search_catalog(df, "batt")[["pn","description_en","description_id","match_score"]]

## 3. Check matching for Indonesian, English and typos

In [ ]:
for query in ["battery", "baterai", "ban", "tirre", "hydraulic filter"]:
    found=search_catalog(df, query)
    print(f"\n{query!r}: {len(found)} result(s)")
    display(found[["pn","description_en","description_id","match_score"]].head(5))

## 4. Prevent strict duplicates and preserve near matches for review

The demo key follows the rule **PN + description + unit**. Only an exact normalized match is blocked; the same PN with a changed unit or description remains visible for human review.

In [ ]:
def duplicate_fingerprint(pn, description, unit):
    return tuple(normalize(v) for v in (pn, description, unit))

seen={}
for idx, row in df.iterrows():
    seen.setdefault(duplicate_fingerprint(row.pn,row.description_en,row.unit), []).append(idx)

new_record={"pn":"demo-bat-001","description":"Forklift battery 48V 600Ah","unit":"Forklift A"}
key=duplicate_fingerprint(new_record["pn"],new_record["description"],new_record["unit"])
print("Exact duplicate?", key in seen)

near_record={"pn":"DEMO-BAT-001","description":"Forklift battery 48V 600Ah","unit":"Forklift B"}
print("Same PN but different unit fingerprint?", duplicate_fingerprint(near_record["pn"],near_record["description"],near_record["unit"]) in seen)

## 5. Flag records that still need confirmation

In [ ]:
required = ["pn", "description_en", "unit", "vendor", "net_price"]
missing = df[required].isna() | df[required].astype(str).apply(lambda col: col.str.strip().eq(""))
df["needs_confirmation"] = missing.any(axis=1)
print("Records:", len(df))
print("Needs confirmation:", int(df.needs_confirmation.sum()))
df.loc[df.needs_confirmation, ["pn","description_en","vendor","net_price","needs_confirmation"]]

## Takeaways and next steps

- Search should consider PN, descriptions in both languages, unit, vendor and brand.
- Bilingual aliases improve recall for common terms, while fuzzy matching can introduce false positives; results need a confidence threshold and user review.
- Exact duplicate keys prevent repeated entry without hiding legitimate variants.
- Blank vendor or price fields should be represented as **needs confirmation**, not as zero or an invented value.

**Scope note:** this notebook is a synthetic portfolio demonstration. It does not reproduce internal source data or claim that the in-app spreadsheet editor is full Microsoft Excel.